## Step 4: Assembly Quality Assessment
**Input:** SPAdes assembly (`04-assembly/spades/option-A/scaffolds.fasta`) 
and scaffolded assembly (`06-scaffolding/ragout_maf_output/`)  
**Output:** QUAST reports in `05-quast-evaluation/`; 
BUSCO reports in `05-busco-evaluation/`  
**Tools:** QUAST v5.3.0, BUSCO v6.0.0 (Ascomycota_odb12 / fungi_odb12)  
**Key parameters:** `--fungus` mode; reference genome: Fo47 (GCF_013085055.1)  
**Key finding:** Post-scaffolding: 12 contigs; N50 = 3.52 Mb; 
BUSCO completeness 95% (94.5% single-copy, 0.5% duplicated)  
**Reference:** Materials & Methods Sections 4.2 & 4.3 — Nebli et al. (2025)

In [ ]:
export SN=3RR
export NCPUS=64

In [ ]:
alias quast.py="apptainer run docker://staphb/quast quast.py"
apptainer build --sandbox busco_sandbox docker://ezlabgva/busco:v6.0.0_cv1
alias busco="apptainer exec --writable busco_sandbox busco"

In [ ]:
mkdir -p 05-quast-evaluation
mkdir -p 05-busco-evaluation

# reference-free mode

In [ ]:
quast.py \
  --fungus \
  -1 02-primary/merged/${SN}-A-illumina_R1.fastq \
  -2 02-primary/merged/${SN}-A-illumina_R2.fastq \
  --nanopore 02-primary/trimmed/${SN}-nanopore.fastq \
  -o 05-quast-evaluation/ \
  -t $NCPUS \
  --labels "spades_A" \
  04-assembly/spades/option-A/scaffolds.fasta

# BUSCO Evaluation

In [ ]:
busco --download fungi_odb12 --download_path 05-busco-evaluation/datasets

In [ ]:
for assembly in 04-assembly/spades/option-A/scaffolds.fasta; do
  label=$(basename $(dirname $assembly))_$(basename $(dirname $(dirname $assembly)))
  busco -i $assembly -o 05-busco-evaluation/${label} --download_path A-05-busco-evaluation/datasets -l fungi_odb12 -m genome -c $NCPUS --force
done

# Genome assembly quality check after secondary scaffolding (Step 5)

In [ ]:
cd 06-scaffolding/evaluation
quast.py \
  --fungus \
  -r 05-quast-evaluation/GCF_013085055.1_ASM1308505v1_genomic.fna \
  -1 02-primary/merged/${SN}-A-illumina_R1.fastq \
  -2 02-primary/merged/${SN}-A-illumina_R2.fastq \
  --nanopore 02-primary/trimmed/${SN}-nanopore.fastq \
  -o 06-scaffolding/evaluation \
  -t $NCPUS \
  --labels "spades_sec_scaffolding" \
  06-scaffolding/ragout_maf_output/sample10_contigs_scaffolds.fasta \

In [ ]:
for assembly in 06-scaffolding/ragout_maf_output/sample10_contigs_scaffolds.fasta ; do
  label=$(basename $(dirname $assembly))_$(basename $(dirname $(dirname $assembly)))
  busco -i $assembly -o 05-busco-evaluation/${label} --download_path 05-busco-evaluation/datasets -l fungi_odb10 -m genome -c $NCPUS --force
done

In [ ]:
for assembly in 06-scaffolding/ragout_maf_output/sample10_contigs_scaffolds.fasta ; do
  label=$(basename $(dirname $assembly))_$(basename $(dirname $(dirname $assembly)))
  busco -i $assembly -o 05-busco-evaluation/${label} --download_path 05-busco-evaluation/datasets -l fungi_odb12 -m genome -c $NCPUS --force
done